# SA-DMAE Segmentation Fine-tuning

BraTS 2021 종양 segmentation (WT / TC / ET)  
SA-DMAE pre-trained encoder (frozen) → segmentation head fine-tuning

> 런타임 → 런타임 유형 변경 → **GPU (T4)** 먼저 설정!

In [ ]:
# ── Cell 1: GPU 확인 & Drive 마운트 ──────────────────────────────────────────
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: 코드 & 패키지 ────────────────────────────────────────────────────
import os

REPO_DIR = '/content/SA-DMAE'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/whkim4338/SA-DMAE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install timm nibabel tensorboard tqdm -q
print('완료')

In [ ]:
# ── Cell 3: 경로 설정 (여기만 수정) ─────────────────────────────────────────

BRATS_PT     = '/content/drive/MyDrive/SA_DMAE_slices/brats'   # ← 수정
BRATS_NIFTI  = '/content/drive/MyDrive/BraTS2021'              # ← 수정
SA_DMAE_CKPT = '/content/drive/MyDrive/SA_DMAE_output/checkpoint-best.pth'
OUTPUT_SA    = '/content/drive/MyDrive/SA_DMAE_seg/sa_dmae'
SEG_CKPT     = f'{OUTPUT_SA}/checkpoint-best.pth'   # 학습 완료 후 자동 생성

EPOCHS = 50; BATCH_SIZE = 16; LR = 1e-3
VAL_RATIO = 0.2; PATIENCE = 15; SAVE_EVERY = 10

import os
os.makedirs(OUTPUT_SA, exist_ok=True)

from pathlib import Path
print(f'BraTS .pt   : {len(list(Path(BRATS_PT).glob("*.pt")))}개')
print(f'SA-DMAE ckpt: {Path(SA_DMAE_CKPT).exists()}')

In [ ]:
# ── Cell 4: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_SA}

In [ ]:
# ── Cell 5: SA-DMAE Segmentation Fine-tuning ──────────────────────────────────
!python main_finetune_seg.py \
    --pt_dir      {BRATS_PT} \
    --nifti_dir   {BRATS_NIFTI} \
    --resume      {SA_DMAE_CKPT} \
    --n_slices    3 --axial_depth 2 \
    --epochs      {EPOCHS} --batch_size {BATCH_SIZE} \
    --lr          {LR} --val_ratio {VAL_RATIO} \
    --patience    {PATIENCE} --save_every {SAVE_EVERY} \
    --output_dir  {OUTPUT_SA} --log_dir {OUTPUT_SA} \
    --device cuda --num_workers 2

In [ ]:
# ── Cell 5-R: 이어서 학습 ────────────────────────────────────────────────────
import glob, os
ckpts = sorted(
    [f for f in glob.glob(f'{OUTPUT_SA}/checkpoint-*.pth') if 'best' not in f and 'final' not in f],
    key=os.path.getmtime
)
if ckpts:
    latest = ckpts[-1]
    print(f'이어서 학습: {latest}')
    !python main_finetune_seg.py \
        --pt_dir {BRATS_PT} --nifti_dir {BRATS_NIFTI} \
        --resume {SA_DMAE_CKPT} --resume_seg {latest} \
        --n_slices 3 --axial_depth 2 \
        --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} \
        --val_ratio {VAL_RATIO} --patience {PATIENCE} --save_every {SAVE_EVERY} \
        --output_dir {OUTPUT_SA} --log_dir {OUTPUT_SA} \
        --device cuda --num_workers 2
else:
    print('저장된 체크포인트 없음.')

In [ ]:
# ── Cell 6: 정량 결과 ─────────────────────────────────────────────────────────
import json, matplotlib.pyplot as plt

logs = [json.loads(l) for l in open(f'{OUTPUT_SA}/log_seg.txt')]
best = max(logs, key=lambda d: d['dice_mean'])

print('=' * 50)
print('SA-DMAE Segmentation (Best Val Dice)')
print('=' * 50)
print(f'  WT  (Whole Tumor)    : {best["dice_wt"]:.4f}')
print(f'  TC  (Tumor Core)     : {best["dice_tc"]:.4f}')
print(f'  ET  (Enhancing Tumor): {best["dice_et"]:.4f}')
print(f'  Mean Dice            : {best["dice_mean"]:.4f}')
print(f'  Best epoch           : {best["epoch"]+1}')
print('=' * 50)

epochs = [d['epoch']+1 for d in logs]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SA-DMAE Segmentation Fine-tuning', fontsize=12)
axes[0].plot(epochs, [d['val_loss']  for d in logs], color='tab:red', linewidth=2)
axes[0].set_title('Val Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [d['dice_wt']   for d in logs], label='WT',   linewidth=2)
axes[1].plot(epochs, [d['dice_tc']   for d in logs], label='TC',   linewidth=2)
axes[1].plot(epochs, [d['dice_et']   for d in logs], label='ET',   linewidth=2)
axes[1].plot(epochs, [d['dice_mean'] for d in logs], label='Mean', linewidth=2, linestyle='--', color='black')
axes[1].set_title('Val Dice'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_SA}/seg_result.png', dpi=120)
plt.show()

In [ ]:
# ── Cell 7: WT 시각화 (Original / GT / Prediction 오버레이) ──────────────────
import torch, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from functools import partial
import torch.nn as nn

import models_sa_dmae, models_seg
from dataset_seg import BraTSSegDataset
from models_seg import SADMAESegmentation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── 모델 로드 ─────────────────────────────────────────────────────────────────
encoder = models_sa_dmae.sa_dmae_vit_base_patch16(n_slices=3, axial_depth=2)
model   = SADMAESegmentation(encoder, num_classes=3, freeze_encoder=True).to(device)

seg_ckpt = torch.load(SEG_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(seg_ckpt['model'])
model.eval()
print('체크포인트 로드 완료')

# ── Val 데이터셋 ──────────────────────────────────────────────────────────────
val_ds = BraTSSegDataset(BRATS_PT, BRATS_NIFTI, split='val', val_ratio=VAL_RATIO)
print(f'Val samples: {len(val_ds)}')

# ── Dice 높은 샘플 Top-N 선택 ──────────────────────────────────────────────────
N_SHOW = 5   # 보여줄 샘플 수

scored = []
with torch.no_grad():
    for i in range(len(val_ds)):
        x, seg = val_ds[i]
        logits  = model(x.unsqueeze(0).to(device))        # (1, 3, 224, 224)
        pred_wt = (logits[0, 0].sigmoid() > 0.5).float()  # WT channel
        gt_wt   = seg[0]                                   # WT GT

        inter = (pred_wt.cpu() * gt_wt).sum()
        union = pred_wt.cpu().sum() + gt_wt.sum()
        dice  = (2 * inter + 1e-5) / (union + 1e-5)
        scored.append((dice.item(), i))

# Dice 기준 내림차순 정렬 → Top-N
scored.sort(key=lambda x: -x[0])
top_indices = [idx for _, idx in scored[:N_SHOW]]
print(f'Top-{N_SHOW} WT Dice: {[f"{scored[i][0]:.3f}" for i in range(N_SHOW)]}')

# ── 시각화 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(N_SHOW, 3, figsize=(10, 3.5 * N_SHOW))
fig.suptitle('SA-DMAE Segmentation — Whole Tumor (WT)', fontsize=13, y=1.01)

col_titles = ['MRI (T1ce)', 'Ground Truth (WT)', 'SA-DMAE Prediction']
for j, title in enumerate(col_titles):
    axes[0][j].set_title(title, fontsize=11)

for row, idx in enumerate(top_indices):
    x, seg = val_ds[idx]
    center  = x[1, 0].numpy()         # center slice, T1ce channel
    gt_wt   = seg[0].numpy()          # WT GT

    with torch.no_grad():
        logits  = model(x.unsqueeze(0).to(device))
        pred_wt = (logits[0, 0].sigmoid() > 0.5).float().cpu().numpy()

    dice_val = scored[row][0]

    # Col 0: MRI + GT 오버레이
    axes[row][0].imshow(center, cmap='gray', vmin=0, vmax=1)
    axes[row][0].contour(gt_wt,   colors='lime',   linewidths=1.2)
    axes[row][0].set_ylabel(f'Dice={dice_val:.3f}', fontsize=9)
    axes[row][0].axis('off')

    # Col 1: GT mask
    axes[row][1].imshow(center, cmap='gray', vmin=0, vmax=1)
    axes[row][1].imshow(gt_wt,  cmap='Greens', alpha=0.5, vmin=0, vmax=1)
    axes[row][1].axis('off')

    # Col 2: Prediction mask
    axes[row][2].imshow(center,  cmap='gray', vmin=0, vmax=1)
    axes[row][2].imshow(pred_wt, cmap='Reds',   alpha=0.5, vmin=0, vmax=1)
    axes[row][2].axis('off')

plt.tight_layout()
save_path = f'{OUTPUT_SA}/seg_visualization.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'저장 완료: {save_path}')